# 📋 Assignment: Automated Payment Reconciliation

**Topic:** Payment Reconciliation using Python & Pandas  
**Total Problems:** 10  

---

## 📌 Instructions

- **Run the Setup Cell first** (Section 0) before attempting any problem. It generates all the data you will need.
- Each problem has a **hints section** — use it if you are stuck, but try without hints first.
- Problems 1–6 are based directly on concepts covered in class.
- Problems 7–10 require you to go one step further and apply the same ideas in new ways.
- You are allowed to use ChatGPT or documentation to look up specific functions, but the **logic and thinking must be your own**.

---

## ⚙️ Section 0: Setup — Run This First!

This cell generates all the data used throughout the assignment.  
**Do not modify this cell.** Just run it and move on to Problem 1.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# ── Configuration ──────────────────────────────────────────
num_transactions = 100
start_date = datetime(2024, 1, 15)

# ── Generate Transaction IDs ────────────────────────────────
transaction_ids = []
for i in range(1001, 1001 + num_transactions):
    txn_id = "TXN" + str(i).zfill(5)
    transaction_ids.append(txn_id)

# ── Generate Amounts ────────────────────────────────────────
amount_options = [299, 499, 799, 999, 1499, 1999, 2499, 3999]
amounts = np.random.choice(amount_options, size=num_transactions)

# ── Generate Timestamps ─────────────────────────────────────
timestamps = []
for i in range(num_transactions):
    random_days = random.randint(0, 2)
    random_hours = random.randint(9, 21)
    random_minutes = random.randint(0, 59)
    timestamp = start_date + timedelta(days=random_days, hours=random_hours, minutes=random_minutes)
    timestamps.append(timestamp)

# ── Generate Payment Methods ────────────────────────────────
method_options = ['UPI', 'Card', 'NetBanking', 'Wallet']
method_weights = [0.5, 0.3, 0.1, 0.1]
payment_methods = np.random.choice(method_options, size=num_transactions, p=method_weights)

# ── Build Internal Ledger ───────────────────────────────────
internal_ledger = pd.DataFrame({
    'transaction_id': transaction_ids,
    'amount': amounts,
    'timestamp': timestamps,
    'payment_method': payment_methods,
    'status': 'SUCCESS'
})

# ── Build Gateway Export (with injected issues) ─────────────
gateway_export = internal_ledger.copy()

# Issue 1: Remove 5 transactions (Missing in Gateway)
missing_indices = np.random.choice(gateway_export.index, size=5, replace=False)
missing_txn_ids = [internal_ledger.loc[idx, 'transaction_id'] for idx in missing_indices]
gateway_export = gateway_export.drop(missing_indices)

# Issue 2: Modify 3 transaction amounts (Amount Mismatch)
mismatch_indices = np.random.choice(gateway_export.index, size=3, replace=False)
fee_options = [10, 20, 50, 100]
for idx in mismatch_indices:
    original_amount = gateway_export.loc[idx, 'amount']
    fee_deducted = random.choice(fee_options)
    gateway_export.loc[idx, 'amount'] = original_amount - fee_deducted

# Issue 3: Add 2 extra transactions (Missing in Ledger)
extra_transactions = pd.DataFrame([
    {'transaction_id': 'TXN99901', 'amount': 1599, 'timestamp': start_date + timedelta(hours=14), 'payment_method': 'UPI', 'status': 'SUCCESS'},
    {'transaction_id': 'TXN99902', 'amount': 2999, 'timestamp': start_date + timedelta(days=1, hours=10), 'payment_method': 'Card', 'status': 'SUCCESS'}
])
gateway_export = pd.concat([gateway_export, extra_transactions], ignore_index=True)

# Issue 4: Change 2 transaction statuses (Status Mismatch)
available_indices = gateway_export.index[:-2].tolist()
status_change_indices = np.random.choice(available_indices, size=2, replace=False)
for idx in status_change_indices:
    gateway_export.loc[idx, 'status'] = 'FAILED'

print("✅ Setup complete!")
print(f"   Internal Ledger : {len(internal_ledger)} transactions")
print(f"   Gateway Export  : {len(gateway_export)} transactions")
print("\nYou can now start solving the problems below.")

✅ Setup complete!
   Internal Ledger : 100 transactions
   Gateway Export  : 97 transactions

You can now start solving the problems below.


---
## 🟢 Problem 1: Explore the Data

Before doing any reconciliation, always inspect your data first.

**Tasks:**
1. Print the **first 5 rows** of `internal_ledger`
2. Print the **first 5 rows** of `gateway_export`
3. Print the **total number of transactions** in each
4. Print the **column names** of both DataFrames

**Expected Output:**  
You should see that `internal_ledger` has **100 transactions** and `gateway_export` has **97 transactions**, with the same 5 columns in both.

> 💡 **Hint:** Use `.head()` to print rows, `len()` for count, and `.columns` for column names.

In [ ]:
# Problem 1 — Your code here
print(internal_ledger)

print(gateway_export)
print(internal_ledger.head(5))
print(len(internal_ledger))
print(internal_ledger.columns)

   transaction_id  amount           timestamp payment_method   status
0        TXN01001    2499 2024-01-17 10:01:00         Wallet  SUCCESS
1        TXN01002     999 2024-01-17 13:15:00           Card  SUCCESS
2        TXN01003    1499 2024-01-15 11:47:00         Wallet  SUCCESS
3        TXN01004    2499 2024-01-15 19:47:00     NetBanking  SUCCESS
4        TXN01005     799 2024-01-17 10:37:00           Card  SUCCESS
..            ...     ...                 ...            ...      ...
95       TXN01096    1999 2024-01-17 11:46:00            UPI  SUCCESS
96       TXN01097     499 2024-01-17 18:30:00           Card  SUCCESS
97       TXN01098     999 2024-01-15 21:30:00           Card  SUCCESS
98       TXN01099    1999 2024-01-16 12:06:00            UPI  SUCCESS
99       TXN01100    1499 2024-01-15 19:27:00            UPI  SUCCESS

[100 rows x 5 columns]
   transaction_id  amount           timestamp payment_method   status
0        TXN01001    2499 2024-01-17 10:01:00         Wallet  SUCC

---
## 🟢 Problem 2: Payment Method Analysis

The finance team wants to understand the transaction breakdown before investigating discrepancies.

**Tasks — using `internal_ledger`:**
1. Find the **count of transactions per payment method**
2. Find the **total amount transacted per payment method**
3. Print which payment method has the **highest total transaction value**

**Expected Output:**  
UPI should have the highest count. Check which method wins on total value.

> 💡 **Hint:** Use `.value_counts()` for counts. Use `.groupby('payment_method')['amount'].sum()` for totals.

In [ ]:
payment_method_counts = internal_ledger['payment_method'].value_counts()
print(payment_method_counts)

payment_totals = internal_ledger.groupby('payment_method')['amount'].sum()
print(payment_totals)

high_payment_method = payment_totals.idxmax()
highest_total_value = payment_totals.max()
print({high_payment_method}, {highest_total_value})

payment_method
UPI           52
Card          25
NetBanking    14
Wallet         9
Name: count, dtype: int64
payment_method
Card          36475
NetBanking    18886
UPI           81948
Wallet        19791
Name: amount, dtype: int64
{'UPI'} {81948}


---
## 🟢 Problem 3: Find Transactions Missing in Gateway

Some transactions are recorded in our internal ledger but never reached the bank's gateway.

**Tasks:**
1. Create a **set** of all transaction IDs from `internal_ledger`
2. Create a **set** of all transaction IDs from `gateway_export`
3. Use **set subtraction** to find IDs present in the ledger but NOT in the gateway
4. Print the **count** of missing transactions
5. Print the **total value at risk** (sum of amounts for these transactions, using `internal_ledger`)

**Expected Output:**  
You should find **5 transactions** missing in the gateway.

> 💡 **Hint:** Set subtraction in Python: `set_a - set_b`. Use `.isin()` to filter the DataFrame.

In [ ]:
# Problem 3 — Your code here
internal_ledger_id = set(internal_ledger['transaction_id'])
gateway_export_id = set(gateway_export['transaction_id'])
missing_id = internal_ledger_id - gateway_export_id


print(f"Total number of missing transactions : {len(missing_id)}")

print(missing_id)

missing_in_gateway = list(missing_id)
missing_in_gateway_df = internal_ledger[internal_ledger['transaction_id'].
                                        isin(missing_in_gateway)].copy()
total_missing_value = missing_in_gateway_df['amount'].sum()

print(f"Total amount at risk : {total_missing_value}")

Total number of missing transactions : 5
{'TXN01058', 'TXN01065', 'TXN01010', 'TXN01027', 'TXN01051'}
Total amount at risk : 7795


---
## 🟢 Problem 4: Find Transactions Missing in Ledger

The bank processed some transactions that our internal system has no record of.

**Tasks:**
1. Use **set subtraction** (reversed this time) to find IDs present in `gateway_export` but NOT in `internal_ledger`
2. Print the **transaction IDs** that are missing in the ledger
3. Fetch and display the **full details** of these transactions from `gateway_export`
4. Print the **total unrecorded value**

**Expected Output:**  
You should find **2 transactions** (`TXN99901` and `TXN99902`) missing in the ledger.

> 💡 **Hint:** This is the reverse of Problem 3. Swap `ledger_ids` and `gateway_ids` in the subtraction.

In [ ]:
# Problem 4 — Your code here
missing_in_ledger = gateway_export_id - internal_ledger_id

print(f"Total no of id missing in ledger : {missing_in_ledger}")
print(f"Missing Transactions : {missing_in_ledger}")


missing_in_ledger_list = list(missing_in_ledger)

missing_in_ledger_df = gateway_export[gateway_export['transaction_id'].
                                      isin(missing_in_ledger_list)].copy()


missing_in_ledger_df['issue_type'] = 'MISSING IN LEDGER'

total_missing_in_ledger = missing_in_ledger_df['amount'].sum()

print(f"Total unrecorded value : {total_missing_in_ledger}")

missing_in_ledger_df

Total no of id missing in ledger : {'TXN99902', 'TXN99901'}
Missing Transactions : {'TXN99902', 'TXN99901'}
Total unrecorded value : 4598


,transaction_id,amount,timestamp,payment_method,status,issue_type
95,TXN99901,1599,2024-01-15 14:00:00,UPI,SUCCESS,MISSING IN LEDGER
96,TXN99902,2999,2024-01-16 10:00:00,Card,SUCCESS,MISSING IN LEDGER


---
## 🟢 Problem 5: Find Amount Mismatches

For transactions present in both systems, the amounts do not always match.

**Tasks:**
1. Find all **common transaction IDs** (present in both ledger and gateway) using set intersection (`&`)
2. Filter both DataFrames to keep only common transactions
3. **Merge** the two filtered DataFrames on `transaction_id` (use `suffixes=('_ledger', '_gateway')`)
4. Add a new column `amount_diff` = `amount_ledger` - `amount_gateway`
5. Filter rows where `amount_diff != 0` and display them
6. Print the **total discrepancy amount**

**Expected Output:**  
You should find **3 transactions** with amount mismatches, totalling ₹250.

> 💡 **Hint:** Use `ledger_ids & gateway_ids` for intersection. Use `pd.merge()` to join.

In [ ]:
# Problem 5 — Your code here
common_transaction_id = internal_ledger_id & gateway_export_id
print(f"Total number of common id's : {len(common_transaction_id)}" )

common_ids_list = list(common_transaction_id)

ledger_common = internal_ledger[internal_ledger['transaction_id'].
                                isin(common_transaction_id)].copy()
gateway_common = gateway_export[gateway_export['transaction_id'].
                                isin(common_ids_list)].copy()

comparison = ledger_common.merge(gateway_common[['transaction_id', 'amount', 'status']],
                                 on='transaction_id',
                                suffixes=('_ledger','_gateway'))

comparison['amount_diff'] = comparison['amount_ledger'] - comparison['amount_gateway']

amount_diff = comparison[comparison['amount_diff'] != 0].copy()

if len(amount_diff) > 0 :
  total_discrepancy = amount_diff['amount_diff'].sum()
  print(f"Total Discrepancy : {total_discrepancy}")


  print("\n Mismatch Details ")
  print("="*60)

  print(amount_diff[['transaction_id', 'amount_ledger', 'amount_gateway',
                     'amount_diff']])
  print("="*60)

  print(f"Total amount mismatch is Rs. {amount_diff['amount_diff'].sum()}")



Total number of common id's : 95
Total Discrepancy : 250

 Mismatch Details 
   transaction_id  amount_ledger  amount_gateway  amount_diff
4        TXN01005            799             749           50
10       TXN01012           2499            2399          100
40       TXN01043            499             399          100
Total amount mismatch is Rs. 250


---
## 🟢 Problem 6: Find Status Mismatches

Our system shows a transaction as `SUCCESS`, but the bank has marked it `FAILED`.

**Tasks** (use the merged comparison table you built in Problem 5):
1. Filter rows where `status_ledger != status_gateway`
2. Display the **transaction ID, ledger amount, ledger status, and gateway status** for each mismatch
3. Print the **count** of status mismatches found

**Expected Output:**  
You should find **2 transactions** where the ledger says `SUCCESS` but the gateway says `FAILED`.

> 💡 **Hint:** You already have the `comparison` DataFrame from Problem 5 with `status_ledger` and `status_gateway` columns. Filter using `!=`.

In [ ]:
# Problem 6 — Your code here
status_mismatches = comparison[comparison['status_ledger'] !=
                               comparison['status_gateway']].copy()
status_mismatches
status_mis = status_mismatches[
    ['transaction_id','amount_ledger', 'status_gateway']
    ].copy()

status_mis


,transaction_id,amount_ledger,status_gateway
85,TXN01091,3999,FAILED
94,TXN01100,1499,FAILED


---
## 🟡 Problem 7: Build the Final Reconciliation Report

Now that you have found all the issues individually, combine them into one clean report — the kind you would hand to a manager.

**Tasks:**
1. For each of the three issue types (missing in gateway, missing in ledger, amount mismatch), create a **separate report DataFrame** with exactly these columns: `transaction_id`, `ledger_amount`, `gateway_amount`, `difference`, `issue_type`
2. Use `pd.concat()` to combine all three into a single `reconciliation_report` DataFrame
3. Sort the report by **absolute value of `difference`** in descending order (biggest issues first)
4. Reset the index so it starts from 0
5. Print the **count of issues per type** using `.value_counts()`
6. Display the final report

**Expected Output:**  
A clean 10-row table sorted from highest to lowest financial impact.

> 💡 **Hint:** For missing-in-gateway rows, `gateway_amount = 0` and `difference = ledger_amount`. For missing-in-ledger rows, `ledger_amount = 0` and `difference = -gateway_amount`. Use `.abs()` for sorting by absolute value.

In [ ]:
# Problem 7 — Your code here
amount_diff.head()
amount_mismatch_report = amount_diff[['transaction_id','amount_ledger',
                                           'amount_gateway','amount_diff']].copy()
amount_mismatch_report['issue_type'] = 'AMOUNT_MISMATCH'
amount_diff.head()
amount_mismatch_report.head()
amount_mismatch_report.columns = ['transaction_id', 'ledger_amount',
                                  'gateway_amount', 'difference', 'issue_type']
print(f"Amount mismatch records: {len(amount_mismatch_report)}")


missing_gateway_report = missing_in_gateway_df[['transaction_id','amount']].copy()
missing_gateway_report['gateway_amount'] = 0
missing_gateway_report['difference'] = missing_gateway_report['amount']
missing_gateway_report['issue_type'] = 'MISSING_GATEWAY'
missing_gateway_report.columns = ['transcation_id','ledger_amount',
                                  'gateway_amount','difference','issue_type']
print(f"Missing in Gateway records : {len(missing_gateway_report)}")
missing_gateway_report.head()

missing_ledger_report = missing_in_ledger_df[['transaction_id','amount']].copy()
missing_ledger_report['ledger_amount'] = 0
missing_ledger_report['gateway_amount'] = missing_ledger_report['amount']
missing_ledger_report['difference'] = missing_ledger_report['amount']
missing_ledger_report['issue_type'] = 'MISSING_IN_LEDGER'

missing_ledger_report = missing_ledger_report[['transaction_id','ledger_amount',
                                               'gateway_amount','difference',
                                               'issue_type']]
print(f" Missing in ledger records : {len(missing_ledger_report)}")


reconciliation_report = pd.concat([
    missing_gateway_report,
    missing_ledger_report,
    amount_mismatch_report
],ignore_index = True)


print(f"\n=== RECONCILIATION REPORT ===")
print(f"Total issues found: {len(reconciliation_report)}")

print(f"\n No. of Transaction by Issue Type : ")
print(reconciliation_report['issue_type'].value_counts())


# Sort by absolute difference (biggest issues first)
reconciliation_report['abs_difference'] = reconciliation_report['difference'].abs()
reconciliation_report = reconciliation_report.sort_values('abs_difference', ascending=False)
reconciliation_report = reconciliation_report.drop('abs_difference', axis=1)

# Reset index for clean display
reconciliation_report = reconciliation_report.reset_index(drop=True)

print("\nFull report (sorted by impact):")
reconciliation_report

Amount mismatch records: 3
Missing in Gateway records : 5
 Missing in ledger records : 2

=== RECONCILIATION REPORT ===
Total issues found: 10

 No. of Transaction by Issue Type : 
issue_type
MISSING_GATEWAY      5
AMOUNT_MISMATCH      3
MISSING_IN_LEDGER    2
Name: count, dtype: int64

Full report (sorted by impact):


,transcation_id,ledger_amount,gateway_amount,difference,issue_type,transaction_id
0,NaN,0,2999,2999,MISSING_IN_LEDGER,TXN99902
1,TXN01051,2499,0,2499,MISSING_GATEWAY,NaN
2,TXN01065,2499,0,2499,MISSING_GATEWAY,NaN
3,TXN01027,1999,0,1999,MISSING_GATEWAY,NaN
4,NaN,0,1599,1599,MISSING_IN_LEDGER,TXN99901
5,TXN01010,499,0,499,MISSING_GATEWAY,NaN
6,TXN01058,299,0,299,MISSING_GATEWAY,NaN
7,NaN,2499,2399,100,AMOUNT_MISMATCH,TXN01012
8,NaN,499,399,100,AMOUNT_MISMATCH,TXN01043
9,NaN,799,749,50,AMOUNT_MISMATCH,TXN01005


---
## 🟡 Problem 8: High-Value Issue Filter

The finance team only wants to review issues above a certain threshold — smaller discrepancies can wait.

**Tasks** (use your `reconciliation_report` from Problem 7):
1. Filter the report to show only issues where the **absolute value of `difference` is greater than ₹1,000**
2. Print the **count** of such high-value issues
3. Print the **total financial exposure** (sum of absolute differences) for these high-value issues
4. Display the filtered table

> 💡 **Hint:** Use `abs()` on the `difference` column inside your filter condition. For example: `df[df['difference'].abs() > 1000]`

In [ ]:
# Problem 8 — Your code here
# Sort by absolute difference (biggest issues first)
final_report = (
    reconciliation_report[
        reconciliation_report['difference'].abs() > 1000
    ]
    .sort_values(by='difference', key=lambda x: x.abs(), ascending=False)
    [['transaction_id', 'ledger_amount', 'gateway_amount', 'difference']]
)
final_report

,transaction_id,ledger_amount,gateway_amount,difference
0,TXN99902,0,2999,2999
1,NaN,2499,0,2499
2,NaN,2499,0,2499
3,NaN,1999,0,1999
4,TXN99901,0,1599,1599


---
## 🟡 Problem 9: Day-wise Discrepancy Analysis

The operations team wants to know which day had the most payment issues — to see if there was a specific system outage.

**Tasks:**
1. Take the **missing-in-gateway transactions** from `internal_ledger` (the 5 transactions you identified in Problem 3)
2. Extract just the **date** from their `timestamp` column (hint: use `.dt.date`)
3. **Group by date** and count how many missing transactions occurred on each day
4. **Group by date** and find the total amount missing on each day
5. Print which date had the **most missing transactions**

> 💡 **Hint:** Once you have a column of timestamps, `df['timestamp'].dt.date` extracts just the date part. Then use `.groupby()` on that new column.

In [ ]:
#your code here



---
## 🔴 Problem 10: Reconciliation Health Score

Management wants a single number that summarises how clean the payment data is — a **reconciliation health score**.

**Definition:**  
> Health Score = (Number of transactions that matched perfectly / Total transactions in internal ledger) × 100

A transaction **matched perfectly** means:
- It exists in **both** ledger and gateway
- The **amount is identical** in both
- The **status is identical** in both

**Tasks:**
1. From your merged comparison table (Problem 5), count transactions where **amount AND status both match** (i.e., `amount_diff == 0` AND `status_ledger == status_gateway`)
2. Divide by the total count in `internal_ledger` and multiply by 100
3. Round to **2 decimal places**
4. Print the health score with a message like: `"Reconciliation Health Score: XX.XX%"`
5. **Bonus:** Print a one-line interpretation — is this score Good (>95%), Average (85–95%), or Poor (<85%)?

> 💡 **Hint:** Use `&` to combine two conditions in a filter. For example: `df[(df['col1'] == 0) & (df['col2'] == df['col3'])]`. Use `round(value, 2)` for rounding.

In [ ]:
# Problem 10 — Your code here


---
## ✅ You're Done!

Once you have completed all 10 problems, review your work:

- Did each output match the expected result described in the problem?
- Can you explain **why** each step works — not just that it works?
- Could you apply this same logic to a dataset with 10,000 transactions?

If you answered yes to all three, you have understood the session well. 🎯